# [1장 통합 실습] 같은 사내 질문을 두 호출 방식으로 처리하기

세 TODO를 채우고 결과를 확인하세요.

## 문제 상황

사내 문서를 바탕으로 직원의 질문에 답하는 Private LLM 서비스를 준비하고 있습니다. 지금은 문서 검색이나 대화 저장을 붙이기 전에, 안내문 한 개와 질문 한 개를 모델에 보내는 기본 기능을 확인하려고 합니다.

담당자는 OpenAI SDK로 작성한 기존 코드와 LangChain으로 나눈 코드를 함께 검토하고 싶어 합니다. 같은 안내문·질문·답변 원칙을 사용해 두 방식으로 답변을 받고, 입력을 준비하는 부분과 응답을 꺼내는 부분이 어디인지 확인하세요. 실습에서는 가상의 안내문을 OpenAI API에 전달합니다.

## 주어진 입력과 만들 결과

| 이름 | 의미 |
| --- | --- |
| `DOCUMENT` | VPN 프로그램을 다시 실행하고, 계속 실패하면 IT지원팀에 문의하라는 안내문입니다. |
| `QUESTION` | VPN이 계속 연결되지 않는 직원의 질문입니다. |
| `SYSTEM` | 안내문에 있는 내용만 사용하고 두 문장 이내로 답하라는 공통 원칙입니다. |
| `USER_TEMPLATE` | 안내문과 질문을 각각 표시하는 문자열 양식입니다. |

최종 결과는 `direct`, `langchain` 두 키를 가진 딕셔너리입니다. 각 값은 해당 방식으로 받은 답변 문자열입니다. 두 답변의 문장은 달라도 됩니다. 안내문의 조치와 문의처가 질문에 맞게 담겼는지 확인하세요.

## 환경에서 먼저 확인할 것

실행 코드는 현재 Python 경로와 설치된 핵심 패키지 버전을 출력합니다. VS Code에서는 이 경로와 선택한 인터프리터가 프로젝트의 `.venv`를 가리키는지 확인하세요. Colab에서는 노트북 런타임의 Python 경로가 표시됩니다.

- `langchain-core`의 `langchain_core`에는 Prompt, 메시지, Parser 같은 공통 구성 요소가 있습니다.
- `langchain-openai`의 `langchain_openai`는 OpenAI 모델을 LangChain 인터페이스에 연결합니다.
- `openai`는 직접 호출에 사용하는 공식 SDK이고, `python-dotenv`의 `dotenv`는 로컬 `.env`를 읽습니다.
- `langchain`은 에이전트 등 상위 기능을 제공합니다. 이번 코드에서 사용하는 구성 요소는 `langchain-core`와 `langchain-openai`에 있습니다.

## 구현할 부분

시작 코드의 세 TODO를 채우세요. 환경 확인·키 입력·실행 예시는 제공되어 있습니다. 정답 파일은 시작 코드를 대체하는 완성본입니다.

### TODO 1. SDK로 답변 받기

`ask_direct(client, document, question)`은 준비된 OpenAI 클라이언트와 문자열 두 개를 받습니다.

1. `USER_TEMPLATE`에 안내문과 질문을 넣습니다.
2. `system` 메시지에 `SYSTEM`, `user` 메시지에 완성한 문자열을 넣고 이 순서로 전달합니다.
3. `client.chat.completions.create()`로 요청합니다. 모델은 `MODEL_NAME`, `reasoning_effort`는 `"none"`, `max_completion_tokens`는 `600`을 사용합니다.
4. 첫 응답 후보의 메시지 본문을 반환합니다. 응답 객체 자체를 반환하지 마세요.

### TODO 2. LangChain의 세 단계 연결하기

`ask_langchain(model, document, question)`은 준비된 ChatModel과 같은 안내문·질문을 받습니다.

- `ChatPromptTemplate`에서 `system`은 `SYSTEM`, `human`은 `USER_TEMPLATE`을 사용합니다.
- 템플릿에 실제 값을 넣은 메시지 목록을 Model에 전달하고, 받은 응답은 `StrOutputParser`로 변환합니다.
- 관찰할 수 있도록 원본 모델 응답과 최종 문자열을 이 순서의 튜플로 반환합니다.

이 장에서는 각 단계를 차례로 실행합니다. 두 번째 모델 호출에도 첫 번째와 같은 입력이 전달되어야 합니다.

### TODO 3. 두 결과 확인하기

`check_answers(direct_answer, chain_answer)`에서 두 값이 모두 문자열이고 공백만 있는 값이 아닌지 검사하세요. 조건을 만족하지 않으면 `ValueError`를 발생시킵니다. 정상일 때는 원문을 수정하지 않고 `{"direct": ..., "langchain": ...}` 형태로 반환합니다.

답변 두 개가 같은 문장인지 비교하는 조건은 넣지 마세요. 모델 호출은 정상 경로에서 총 두 번이며, 나머지는 환경 확인과 Python 객체 처리입니다.

## 실행 준비

**Windows + VS Code + PowerShell + Python 3.12 + uv**

아래 자료 ZIP을 내려받아 압축을 풀고, `pyproject.toml`이 있는 폴더를 VS Code로 엽니다. Python과 Jupyter 확장을 설치하고 **터미널 → 새 터미널**에서 PowerShell을 선택하세요. uv가 없다면 먼저 `python -m pip install uv`를 실행합니다.

```powershell
uv sync --python 3.12
uv run python --version
uv run python starter.py
```

버전 출력이 `Python 3.12.x`인지 확인합니다. 제공된 `.python-version`과 `pyproject.toml`도 Python 3.12를 지정합니다. 해당 Python이 없으면 uv가 준비합니다. 별도의 가상환경 활성화 명령은 필요하지 않습니다.

노트북으로 풀려면 `langchain_chapter01_starter.ipynb`를 열고 오른쪽 위 **커널 선택**에서 이 폴더의 `.venv`를 선택합니다. `uv sync`를 마쳤다면 노트북의 패키지 설치 셀은 건너뛰고 준비 코드부터 실행하세요.

모델 호출에는 수업용 OpenAI API 키가 필요합니다. 처음 준비할 때 `.env.example`을 `.env`로 복사해 `OPENAI_API_KEY`를 설정하세요. 이미 `.env`가 있으면 덮어쓰지 않습니다. 키를 코드·노트북 본문·실행 결과에 붙여 넣지 않습니다. `OPENAI_MODEL`의 기본값은 `gpt-5.6-luna`입니다. 모델을 바꿀 때는 수업에서 안내한 모델 중 해당 API 키로 사용할 수 있는 모델을 선택하세요. 로컬에서는 `.env`의 `OPENAI_MODEL`을, Colab에서는 준비 셀의 `MODEL_NAME`을 사용할 모델명으로 바꾸고 준비 셀부터 다시 실행합니다.

**선택: Google Colab**

Colab을 사용할 때는 **파일 → 노트북 업로드**에서 `langchain_chapter01_starter.ipynb`를 엽니다. 필요한 데이터는 노트북에 포함되어 있습니다. 첫 패키지 설치 셀부터 실행하고, 이미 불러온 패키지의 버전 변경 안내가 나오면 런타임을 다시 시작합니다.

```python
%pip install -q "langchain-core==1.5.1" "langchain-openai==1.4.1" "openai==2.48.0" "python-dotenv==1.2.2" "pydantic==2.13.5"
```

Colab에서는 실행 중 표시되는 가림 입력창에 키를 입력하세요. 키를 출력하지 않습니다.

시작 코드의 세 TODO는 작성 전 `NotImplementedError`로 멈추도록 되어 있습니다. 각 묶음을 채운 뒤 처음부터 다시 실행하세요.

## 패키지 설치

Windows에서 `uv sync`를 마쳤다면 이 설치 셀은 건너뜁니다. Colab에서는 설치 셀부터 실행하세요.

In [ ]:
%pip install -q "langchain-core==1.5.1" "langchain-openai==1.4.1" "openai==2.48.0" "python-dotenv==1.2.2" "pydantic==2.13.5"


## API 키 준비

모델을 호출하기 전에 수업용 OpenAI API 키를 준비하세요. 처음 실행할 때 가림 입력창이 나타나면 키를 입력합니다. 로컬에서는 같은 폴더의 `.env` 파일에 설정해도 됩니다. 키를 코드나 텍스트 셀에 직접 붙여 넣지 않습니다.

`OPENAI_MODEL`의 기본값은 `gpt-5.6-luna`입니다. 모델을 바꿀 때는 수업에서 안내한 모델 중 해당 API 키로 사용할 수 있는 모델을 선택하세요. 로컬에서는 `.env`의 `OPENAI_MODEL`을, Colab에서는 준비 셀의 `MODEL_NAME`을 사용할 모델명으로 바꾸고 준비 셀부터 다시 실행합니다.

모델의 답변 문장은 실행마다 달라질 수 있으므로 뒤에 안내한 필수 정보와 출력 형식을 확인하세요.

## 준비

In [ ]:
import getpass
import os
import sys
from importlib.metadata import version

from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from openai import OpenAI

load_dotenv()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
# 실제 사내 자료 대신 실습용으로 만든 안내문을 사용합니다.
DOCUMENT = "VPN 연결이 끊기면 프로그램을 다시 실행하세요. 계속 실패하면 IT지원팀에 문의하세요."
QUESTION = "VPN이 계속 연결되지 않으면 어떻게 해야 하나요?"
SYSTEM = "사내 안내 도우미입니다. 제공된 안내문만 근거로 질문에 두 문장 이내로 답하세요."
USER_TEMPLATE = "안내문: {document}\n질문: {question}"



## 구현

In [ ]:
def ask_direct(client, document, question):
    # TODO 1: SDK 메시지를 만들고 첫 응답 후보의 본문을 반환하세요.
    raise NotImplementedError("TODO 1을 구현하세요.")


def ask_langchain(model, document, question):
    # TODO 2: 역할별 메시지, 모델 응답, 문자열을 순서대로 만들고 마지막 두 값을 반환하세요.
    raise NotImplementedError("TODO 2을 구현하세요.")


def check_answers(direct_answer, chain_answer):
    # TODO 3: 두 값의 문자열 여부와 빈 값을 확인하고 direct/langchain 딕셔너리를 반환하세요.
    raise NotImplementedError("TODO 3을 구현하세요.")



## 결과 확인

In [ ]:
def main():
    print("Python:", sys.executable)
    for package in ("langchain-core", "langchain-openai", "openai", "python-dotenv"):
        print(f"{package}: {version(package)}")
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키: ")
    if not os.environ["OPENAI_API_KEY"].strip():
        raise ValueError("API 키가 비어 있습니다.")
    print("모델:", MODEL_NAME)
    # 객체 생성은 요청을 보내지 않습니다. 위 두 함수가 각각 한 번 호출합니다.
    model = ChatOpenAI(model=MODEL_NAME, reasoning_effort="none",
                       max_completion_tokens=600, timeout=30, max_retries=0,
                       use_responses_api=False)
    with OpenAI(timeout=30, max_retries=0) as client:
        direct = ask_direct(client, DOCUMENT, QUESTION)
    message, chained = ask_langchain(model, DOCUMENT, QUESTION)
    results = check_answers(direct, chained)
    assert isinstance(message, AIMessage)
    print("Model 결과가 AIMessage인가:", isinstance(message, AIMessage))
    for name, answer in results.items():
        print(f"[{name}] {answer}")
    print("답변 문자열 확인:", all(isinstance(value, str) for value in results.values()))


if __name__ == "__main__":
    main()


## 결과 확인

Python 경로와 패키지 버전 다음에 아래 형태가 표시됩니다. 답변 문장은 실행마다 달라질 수 있습니다.

```text
모델: gpt-5.6-luna
Model 결과가 AIMessage인가: True
[direct] VPN 프로그램을 다시 실행해 보세요. 계속 실패하면 IT지원팀에 문의하세요.
[langchain] VPN 프로그램을 재실행한 뒤에도 연결되지 않으면 IT지원팀에 문의하세요.
답변 문자열 확인: True
```

모델 이름은 `OPENAI_MODEL` 설정에 따라 달라집니다. 고정적으로 확인할 부분은 두 방식 모두 비어 있지 않은 문자열을 반환하고, LangChain의 Model 단계가 `AIMessage`를 반환한다는 점입니다. 안내문에 없는 재설치 절차나 연락처가 추가되지 않았는지는 답변을 읽고 확인하세요. 문자열 검사는 답변 내용의 사실성까지 판정하지 않습니다.

`ValueError`가 발생하면 반환한 값이 응답 객체인지 문자열인지부터 확인하세요. 인증·모델 접근 오류라면 키와 `OPENAI_MODEL` 설정을 확인합니다. 응답이 비었다면 모델의 응답 본문을 확인하고, 정해진 성공 문구로 대체하지 마세요.

## 실행 후 생각해 볼 것

1. 답변 원칙만 바꾸려면 어느 값을 수정해야 하나요? Retriever나 Memory를 나중에 추가한다면 기본 흐름의 어디에 정보를 더하게 될까요?
2. 두 결과가 다른 문장이라는 이유만으로 한쪽 코드가 틀렸다고 판단할 수 있나요? 단순한 일회성 질문과 여러 단계를 연결할 서비스에서 코드 구조의 선택 기준은 어떻게 달라질까요?

이 장은 약 60분 동안 TODO 구현과 결과 확인, 관찰 메모 작성까지 진행합니다. 완성한 코드 또는 노트북과 같은 입력을 사용한 두 호출의 답변을 제출하세요. SDK 응답에서 꺼낸 문자열과 LangChain의 `AIMessage`→문자열 변환을 짚고, 두 답변이 원문의 안내 순서를 지켰는지 비교합니다. 문장을 똑같이 맞추려고 추가 호출할 필요는 없습니다.

## 나의 관찰 메모

위의 관찰 질문에 대한 답을 이 텍스트 셀에 작성하세요.
